<a href="https://colab.research.google.com/github/kaung-min-thant/SubImpact-AI/blob/main/SubImpact_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [30]:
!pip install statsbombpy scikit-learn xgboost lightgbm shap pandas numpy matplotlib seaborn --quiet

In [31]:
import pandas as pd
import numpy as np
import warnings
import os

from statsbombpy import sb

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report
)

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.4f}".format)

print("All imports successful.")

All imports successful.


In [32]:
COMPETITIONS = [
    {"competition_id": 43, "season_id": 106, "name": "FIFA World Cup 2022"},
]

WINDOW_MIN   = 15
MIN_POST_MIN = 5
IMPACT_POS_THRESHOLD =  0.10
IMPACT_NEG_THRESHOLD = -0.10

OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Config set. Window={WINDOW_MIN}min | Thresholds: +{IMPACT_POS_THRESHOLD} / {IMPACT_NEG_THRESHOLD}")

Config set. Window=15min | Thresholds: +0.1 / -0.1


In [33]:
def get_score_at_minute(events_df, team_name, minute):
    goals = events_df[
        (events_df["type"] == "Shot") &
        (events_df["shot_outcome"] == "Goal") &
        (events_df["minute"] < minute)
    ]
    team_goals = goals[goals["team"] == team_name].shape[0]
    opp_goals  = goals[goals["team"] != team_name].shape[0]
    return team_goals, opp_goals

def get_xg_in_window(events_df, team_name, start_min, end_min):
    window = events_df[
        (events_df["type"] == "Shot") &
        (events_df["minute"] >= start_min) &
        (events_df["minute"] < end_min)
    ].copy()
    team_xg = window[window["team"] == team_name]["shot_statsbomb_xg"].sum()
    opp_xg  = window[window["team"] != team_name]["shot_statsbomb_xg"].sum()
    return round(team_xg, 4), round(opp_xg, 4)

def get_event_counts_in_window(events_df, team_name, start_min, end_min):
    window = events_df[
        (events_df["team"] == team_name) &
        (events_df["minute"] >= start_min) &
        (events_df["minute"] < end_min)
    ]
    shots     = (window["type"] == "Shot").sum()
    passes    = (window["type"] == "Pass").sum()
    pressures = (window["type"] == "Pressure").sum()
    return int(shots), int(passes), int(pressures)

def get_position_group(position_name):
    if not position_name:
        return "Unknown"
    pos = position_name.lower()
    if any(p in pos for p in ["forward", "wing", "striker", "second striker"]):
        return "FW"
    elif any(p in pos for p in ["midfield", "attacking mid", "defensive mid"]):
        return "MF"
    elif any(p in pos for p in ["back", "sweeper", "wing back"]):
        return "DF"
    elif "goalkeeper" in pos or "keeper" in pos:
        return "GK"
    else:
        return "Unknown"

In [34]:
def classify_impact(score):
    if score > IMPACT_POS_THRESHOLD:
        return "Positive"
    elif score < IMPACT_NEG_THRESHOLD:
        return "Negative"
    else:
        return "Neutral"

In [35]:
def get_game_phase(minute):
    if minute <= 30:
        return "0-30"
    elif minute <= 60:
        return "31-60"
    elif minute <= 75:
        return "61-75"
    else:
        return "76-90+"

In [36]:
print("\nLoading match lists from StatsBomb...")
all_matches = []

for comp in COMPETITIONS:
    try:
        matches = sb.matches(
            competition_id=comp["competition_id"],
            season_id=comp["season_id"]
        )
        matches["competition_name"] = comp["name"]
        all_matches.append(matches)
        print(f"   OK: {comp['name']}: {len(matches)} matches")
    except Exception as e:
        print(f"   FAILED: {comp['name']}: {e}")

matches_df = pd.concat(all_matches, ignore_index=True)
print(f"\nTotal matches loaded: {len(matches_df)}")


Loading match lists from StatsBomb...
   OK: FIFA World Cup 2022: 64 matches

Total matches loaded: 64


In [37]:
print("\nExtracting substitution-level features...")
records = []
failed  = 0
match_ids = matches_df["match_id"].tolist()

i = 0
for match_id in match_ids:

    if i % 20 == 0:
        print(f"   Progress: {i}/{len(match_ids)} matches | Samples so far: {len(records)}")
    i += 1

    try:
        events = sb.events(match_id=match_id)

        # Normalize columns
        if "type" in events.columns and events["type"].dtype == object:
            if type(events["type"].iloc[0]) == dict:
                events["type"] = events["type"].apply(lambda x: x.get("name", "") if type(x) == dict else x)

        if "shot_outcome" in events.columns and events["shot_outcome"].dtype == object:
            if type(events["shot_outcome"].dropna().iloc[0]) == dict:
                events["shot_outcome"] = events["shot_outcome"].apply(
                    lambda x: x.get("name", "") if type(x) == dict else x
                )

        if "team" in events.columns and events["team"].dtype == object:
            if type(events["team"].iloc[0]) == dict:
                events["team"] = events["team"].apply(lambda x: x.get("name", "") if type(x) == dict else x)

        if "shot_statsbomb_xg" not in events.columns:
            events["shot_statsbomb_xg"] = 0.0
        events["shot_statsbomb_xg"] = pd.to_numeric(events["shot_statsbomb_xg"], errors="coerce").fillna(0.0)

        subs = events[events["type"] == "Substitution"].copy()
        if subs.empty:
            continue

        match_row  = matches_df[matches_df["match_id"] == match_id].iloc[0]
        home_team  = match_row.get("home_team", {})
        if type(home_team) == dict:
            home_team = home_team.get("home_team_name", "")

        match_minute_max = events["minute"].max()
        sub_seq_tracker = {}

        for _, sub in subs.iterrows():
            sub_min  = int(sub["minute"])
            team     = sub["team"] if type(sub["team"]) == str else sub["team"].get("name", "")

            if not team:
                continue

            time_remaining = 90 - sub_min
            if time_remaining < MIN_POST_MIN:
                continue

            # Extract position of the player going OFF
            raw_position = sub.get("position", None)
            if type(raw_position) == dict:
                position_name = raw_position.get("name", "")
            elif type(raw_position) == str:
                position_name = raw_position
            else:
                position_name = ""
            position_group = get_position_group(position_name)

            sub_seq_tracker[team] = sub_seq_tracker.get(team, 0) + 1
            sub_sequence = sub_seq_tracker[team]

            pre_start = max(0, sub_min - WINDOW_MIN)
            pre_end   = sub_min

            team_xg_prev, opp_xg_prev = get_xg_in_window(events, team, pre_start, pre_end)
            shots_prev, passes_prev, pressures_prev = get_event_counts_in_window(
                events, team, pre_start, pre_end
            )
            team_goals, opp_goals = get_score_at_minute(events, team, sub_min)
            score_diff   = team_goals - opp_goals
            is_home      = 1 if team == home_team else 0
            xg_diff_prev = round(team_xg_prev - opp_xg_prev, 4)

            post_start = sub_min
            post_end   = min(sub_min + WINDOW_MIN, int(match_minute_max) + 1)

            team_xg_next, opp_xg_next = get_xg_in_window(events, team, post_start, post_end)
            xg_diff_next = round(team_xg_next - opp_xg_next, 4)

            impact_score = round(xg_diff_next - xg_diff_prev, 4)
            impact_label = classify_impact(impact_score)

            records.append({
                "match_id":         match_id,
                "competition":      match_row.get("competition_name", ""),
                "team":             team,
                "sub_minute":       sub_min,
                "score_diff":       score_diff,
                "is_home":          is_home,
                "time_remaining":   time_remaining,
                "sub_sequence":     sub_sequence,
                "game_phase":       get_game_phase(sub_min),
                "position_name":    position_name,
                "position_group":   position_group,
                "team_xg_prev15":   team_xg_prev,
                "opp_xg_prev15":    opp_xg_prev,
                "xg_diff_prev15":   xg_diff_prev,
                "shots_prev15":     shots_prev,
                "passes_prev15":    passes_prev,
                "pressures_prev15": pressures_prev,
                "team_xg_next15":   team_xg_next,
                "opp_xg_next15":    opp_xg_next,
                "xg_diff_next15":   xg_diff_next,
                "impact_score":     impact_score,
                "impact_label":     impact_label,
            })

    except Exception as e:
        failed += 1
        continue

print(f"\nExtraction complete!")
print(f"   Total substitution samples: {len(records)}")
print(f"   Failed matches (skipped):   {failed}")


Extracting substitution-level features...
   Progress: 0/64 matches | Samples so far: 0
   Progress: 20/64 matches | Samples so far: 168
   Progress: 40/64 matches | Samples so far: 310
   Progress: 60/64 matches | Samples so far: 469

Extraction complete!
   Total substitution samples: 503
   Failed matches (skipped):   0
